In [22]:
from pathlib import Path

import colorcet as cc
import equinox as eqx
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
from dynamical_systems.analysis import lyapunov_gr
from dynamical_systems.continuous import Lorenz63, Lorenz96
from dynamical_systems.metrics import (
    cosine_similarity as cosine_similarity,
    maximum_mean_discrepancy as maximum_mean_discrepancy,
    root_mean_squared_error as root_mean_squared_error,
    sinkhorn_divergence as sinkhorn_divergence,
)
from dynamics_discovery.data import TimeSeriesDataset
from dynamics_discovery.io import load_model
from dynamics_discovery.models import NeuralODE
from dynamics_discovery.utils.tree import tree_stack
from jaxtyping import Array, Float
from matplotlib_utils.plots import plot_line_and_band


plt.style.use("matplotlib_utils.styles.dash_gridded")
jax.config.update("jax_enable_x64", True)


EXP_TYPES_ALL = (
    "vanilla",
    "neighborhood",
    "efficient_neighborhood",
    "dyslim",
    "dyslim_neigh",
)
EXP_TYPES = (
    "efficient_neighborhood",
    "vanilla",
    # "neighborhood",
    "dyslim",
    # "dyslim_neigh",
)

COLOR_DICT = dict(
    zip(
        EXP_TYPES_ALL,
        (
            cc.cm.glasbey_dark(45),
            cc.cm.glasbey_dark(0),
            cc.cm.glasbey_dark(64),
            cc.cm.glasbey_dark(46),
            cc.cm.glasbey_dark(91),
        ),
    )
)
LABEL_DICT = dict(
    zip(
        EXP_TYPES_ALL,
        (
            "Vanilla",
            "Neighborhood (Ours)",
            "Neighborhood (Ours)",
            "DySLIM (ICML 2024)",
            "DySLIM - Neighborhood",
        ),
    )
)

ODE_DICT = {
    "lorenz63": Lorenz63(),
    "lorenz96_6d": Lorenz96(dim=6),
    "lorenz96_18d": Lorenz96(dim=18),
    "lorenz96_54d": Lorenz96(dim=54),
}

ODE_LABEL_DICT = {
    "lorenz63": "Lorenz63",
    "lorenz96_6d": "Lorenz96 (6D)",
    "lorenz96_18d": "Lorenz96 (18D)",
    "lorenz96_54d": "Lorenz96 (54D)",
}

In [4]:
def load_experiment(
    dataset_name: str,
    exp_type: str,
    key: int,
    noise: float,
    rootdir=Path("../../results/revised/"),
) -> NeuralODE:
    loaddir = rootdir / dataset_name / exp_type / "best"
    model_path_template = f"*_noise={noise}_key={key}*"
    model_paths = list(loaddir.glob(model_path_template))
    if len(model_paths) == 0:
        raise ValueError("No models found with the given conditions")
    elif len(model_paths) > 1:
        raise ValueError("Multiple models found with the given conditions!")
    else:
        return load_model(model_paths[0])


def load_test_dataset(dataset_name: str, rootdir=Path("../../data")):
    standardizer = TimeSeriesDataset.from_hdf5(
        rootdir / f"{dataset_name}_train.hdf5", time_slice=slice(0, 500 * 2, 2)
    ).standardize()[1]
    dataset_test = TimeSeriesDataset.from_hdf5(
        rootdir / f"{dataset_name}_test.hdf5", time_slice=slice(0, 500 * 2, 2)
    ).apply_transform(standardizer)
    return dataset_test, standardizer

In [29]:
LYAPUNOV_TIMES = {
    "lorenz63": (0.0, 400.0, 0.2),
    "lorenz96_6d": (0.0, 125.0, 1.0),
    "lorenz96_18d": (0.0, 125.0, 1.0),
    "lorenz96_54d": (0.0, 125.0, 1.0),
}


def calculate_lyapunovs(
    dataset_name: str,
    exp_types=("efficient_neighborhood", "vanilla", "dyslim"),
    noises=(0.01, 0.05, 0.1),
    keys=(0, 10, 20),
):
    dataset_test, _ = load_test_dataset(dataset_name)

    @eqx.filter_jit
    @eqx.filter_vmap
    def calculate_lyapunov(model) -> Float[Array, " dim"]:
        lyas = eqx.filter_vmap(
            lambda u0_: lyapunov_gr(
                model,
                u0_,
                jnp.arange(*LYAPUNOV_TIMES[dataset_name]),
                rtol=1e-4,
                atol=1e-6,
                max_steps=2048,
            )[0][-1],
        )(dataset_test.u[:, 0])
        return jnp.mean(lyas, axis=0)

    models = {
        e: {n: [load_experiment(dataset_name, e, k, n) for k in keys] for n in noises}
        for e in exp_types
    }
    models_list, treedef = jax.tree.flatten(
        models, is_leaf=lambda x: isinstance(x, eqx.Module)
    )
    models_stacked = tree_stack(models_list)
    lyas: Float[Array, "models dim"] = calculate_lyapunov(models_stacked)

    return jax.tree.unflatten(treedef, lyas), dataset_test.metadata["lyapunovs"]

In [30]:
lyas, lyas_true = calculate_lyapunovs("lorenz96_6d")

In [31]:
lyas_true

array([ 1.24081300e+00,  2.42814245e-03, -1.04331393e-01, -8.48424051e-01,
       -1.61674637e+00, -4.67373933e+00])

In [32]:
lyas

{'dyslim': {0.01: [Array([ 1.23488159,  0.02517527, -0.15636812, -0.86603919, -1.65226076,
          -4.77243019], dtype=float64),
   Array([ 1.21621892,  0.03079297, -0.12575252, -0.86612319, -1.64322615,
          -4.81511015], dtype=float64),
   Array([ 1.27950817,  0.0261678 , -0.15905087, -0.88072178, -1.65051811,
          -4.70667462], dtype=float64)],
  0.05: [Array([ 1.2106637 ,  0.01422664, -0.26049926, -1.01693454, -1.93604644,
          -6.26064165], dtype=float64),
   Array([ 1.28085553,  0.00730918, -0.29789107, -1.01325239, -1.96925631,
          -6.27125947], dtype=float64),
   Array([ 1.23775459,  0.01235677, -0.25316423, -0.97137996, -1.86181578,
          -6.12973242], dtype=float64)],
  0.1: [Array([ 1.18899941,  0.00686104, -0.38284852, -1.05883161, -2.04173822,
          -6.79647641], dtype=float64),
   Array([ 1.25930769e+00,  3.06058490e-03, -3.52002167e-01, -1.04337133e+00,
          -2.02575956e+00, -7.41066899e+00], dtype=float64),
   Array([ 1.19751686,  0.0

In [ ]:
lya_nrmse = jax.tree.map(
    lambda x: {
        "mean": jnp.mean(jnp.asarray(x), axis=0),
        "std": jnp.std(jnp.asarray(x), axis=0),
    },
    jax.tree.map(lambda l: jnp.sqrt((l - lyas_true) ** 2) / lyas_true**2, lyas),
    is_leaf=lambda x: isinstance(x, list),
)
lya_nrmse

{'dyslim': {0.01: {'mean': Array([1.49865549e-02, 4.23186297e+03, 3.92518341e+00, 3.13095359e-02,
          1.22125360e-02, 4.16588808e-03], dtype=float64),
   'std': Array([8.71573539e-03, 4.15211406e+02, 1.38763152e+00, 9.58808463e-03,
          1.49717524e-03, 2.04183400e-03], dtype=float64)},
  0.05: {'mean': Array([1.58589741e-02, 1.50433729e+03, 1.52674497e+01, 2.11299400e-01,
          1.16924947e-01, 7.08119650e-02], dtype=float64),
   'std': Array([1.01540579e-02, 4.95546326e+02, 1.79933403e+00, 2.87035719e-02,
          1.71835010e-02, 2.94636800e-03], dtype=float64)},
  0.1: {'mean': Array([2.45958180e-02, 6.60722246e+02, 2.28745284e+01, 2.93702610e-01,
          1.41541689e-01, 1.11208857e-01], dtype=float64),
   'std': Array([9.17989373e-03, 4.19662288e+02, 2.16705704e+00, 1.92744046e-02,
          2.55679107e-02, 1.14789586e-02], dtype=float64)}},
 'efficient_neighborhood': {0.01: {'mean': Array([1.86647448e-02, 3.64024567e+03, 2.53482219e+00, 1.80831276e-02,
          6.

: 

In [ ]:
jax.tree.unflatten(treedef, (lyas_ - lya_known) ** 2 / lya_known**2)

lya_nrmse = jax.tree.map(
    lambda x: {
        "mean": jnp.mean(jnp.asarray(x), axis=0),
        "std": jnp.std(jnp.asarray(x), axis=0),
    },
    jax.tree.unflatten(treedef, jnp.sqrt((lyas_ - lya_known) ** 2 / lya_known**2)),
    is_leaf=lambda x: isinstance(x, list),
)
lya_nrmse

noises = (0.01, 0.05, 0.1)
fig, axes = plt.subplots(1, 3, figsize=(6, 3), sharex=True, sharey=True)
for ax, n in zip(axes, noises):
    for exp_type in EXP_TYPES:
        plot_line_and_band(
            ax,
            np.arange(3),
            lya_nrmse[exp_type][n]["mean"],
            lya_nrmse[exp_type][n]["std"],
            color=COLOR_DICT[exp_type],
        )
    ax.set_yscale("log")